BOOTCAMP AI MACCHINE LEARNING TRAINEE

MARIO ERNESTO MEJIA TORRES

In [ ]:
import pandas as pd

# 1. Cargar viajes de taxi amarillo de enero de 2024
url_viajes = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
trips = pd.read_parquet(url_viajes)

# 2. Cargar tabla de nombres y distritos de zonas
url_zonas = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zones = pd.read_csv(url_zonas)

# Verificar cuántos viajes cargamos y ver las primeras filas
print(f"Total de viajes cargados: {len(trips):,}")
trips.head()

Total de viajes cargados: 2,964,624


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [ ]:
# 1. Redondear la fecha/hora de inicio del viaje a la hora exacta (.dt.floor("h"))
trips['hora'] = trips['tpep_pickup_datetime'].dt.floor('h')

# 2. Contar viajes por zona de origen (PULocationID)
conteo_por_zona = trips.groupby('PULocationID').size()

# 3. Filtrar zonas con al menos 3,000 viajes
zonas_validas = conteo_por_zona[conteo_por_zona >= 3000].index.tolist()

print(f"📊 De las zonas registradas, {len(zonas_validas)} cumplen con el mínimo de 3,000 viajes.")

📊 De las zonas registradas, 59 cumplen con el mínimo de 3,000 viajes.


In [ ]:
import pandas as pd

# 1. Crear el rango de todas las horas de enero + las 24 horas del 1 de febrero
fechas_completas = pd.date_range("2024-01-01 00:00:00", "2024-02-01 23:00:00", freq="h")

# 2. Combinar cada una de las 59 zonas con todas las horas posibles
index_completo = pd.MultiIndex.from_product(
    [zonas_validas, fechas_completas],
    names=['PULocationID', 'hora']
)

# 3. Contar los viajes reales por zona y hora
conteo_real = trips[trips['PULocationID'].isin(zonas_validas)].groupby(['PULocationID', 'hora']).size()

# 4. Reordenar con la grilla completa y rellenar las horas faltantes con 0
df_series = conteo_real.reindex(index_completo, fill_value=0).reset_index(name='viajes')

print(f"📏 Filas totales en la grilla completa: {len(df_series):,}")
df_series.head()

📏 Filas totales en la grilla completa: 45,312


,PULocationID,hora,viajes
0,4,2024-01-01 00:00:00,25
1,4,2024-01-01 01:00:00,29
2,4,2024-01-01 02:00:00,34
3,4,2024-01-01 03:00:00,31
4,4,2024-01-01 04:00:00,32


In [ ]:
# 1. Asegurar el orden cronológico por zona
df_series = df_series.sort_values(['PULocationID', 'hora']).reset_index(drop=True)

# 2. Agrupar por zona para calcular los lags de cada una de forma independiente
grp = df_series.groupby('PULocationID')['viajes']

# 3. Crear los Lags (valores pasados)
df_series['lag_24'] = grp.shift(24)     # Hace 1 día
df_series['lag_48'] = grp.shift(48)     # Hace 2 días
df_series['lag_167'] = grp.shift(167)   # Hace una semana menos 1 hora
df_series['lag_168'] = grp.shift(168)   # Hace exactamente una semana
df_series['lag_169'] = grp.shift(169)   # Hace una semana más 1 hora

# Promedio diario del día anterior
df_series['media_dia_ant'] = grp.transform(lambda x: x.shift(24).rolling(24).mean())

# 4. Variables de calendario
df_series['num_hora'] = df_series['hora'].dt.hour
df_series['dia_semana'] = df_series['hora'].dt.dayofweek
# 15 de enero de 2024 (Martin Luther King Jr. Day)
df_series['feriado'] = (df_series['hora'].dt.date == pd.to_datetime("2024-01-15").date()).astype(int)
df_series['zona'] = df_series['PULocationID'].astype('category')

print("✅ Variables de series de tiempo creadas con éxito.")
df_series.tail()

✅ Variables de series de tiempo creadas con éxito.


,PULocationID,hora,viajes,lag_24,lag_48,lag_167,lag_168,lag_169,media_dia_ant,num_hora,dia_semana,feriado,zona
45307,264,2024-02-01 19:00:00,0,18.0,18.0,20.0,20.0,21.0,12.791667,19,3,0,264
45308,264,2024-02-01 20:00:00,0,13.0,17.0,19.0,20.0,20.0,12.625000,20,3,0,264
45309,264,2024-02-01 21:00:00,0,12.0,24.0,19.0,19.0,20.0,12.125000,21,3,0,264
45310,264,2024-02-01 22:00:00,0,15.0,14.0,17.0,19.0,19.0,12.166667,22,3,0,264
45311,264,2024-02-01 23:00:00,0,4.0,6.0,9.0,17.0,19.0,12.083333,23,3,0,264


In [ ]:
# 1. Eliminar filas con valores vacíos causados por los lags al inicio del mes
df_limpio = df_series.dropna().copy()

# 2. Separar por rangos de fechas estrictos
train = df_limpio[(df_limpio['hora'] >= "2024-01-08") & (df_limpio['hora'] <= "2024-01-24 23:00:00")]
test = df_limpio[(df_limpio['hora'] >= "2024-01-25") & (df_limpio['hora'] <= "2024-01-31 23:00:00")]
futuro = df_series[df_series['hora'] >= "2024-02-01"]

print(f"🏋️ Filas de Entrenamiento: {len(train):,}")
print(f"🧪 Filas de Prueba (Test): {len(test):,}")
print(f"🔮 Filas para el 'Mañana' (1 Feb): {len(futuro):,}")

🏋️ Filas de Entrenamiento: 24,013
🧪 Filas de Prueba (Test): 9,912
🔮 Filas para el 'Mañana' (1 Feb): 1,416


In [ ]:
import xgboost as xgb

# 1. Definir las variables explicativas (Features) y la variable a predecir (Target)
FEATURES = ['lag_24', 'lag_48', 'lag_167', 'lag_168', 'lag_169', 'media_dia_ant', 'num_hora', 'dia_semana', 'feriado', 'zona']
TARGET = 'viajes'

# 2. Definir los 9 cuantiles que necesitamos (10%, 20%, ..., 90%)
cuantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# 3. Configurar el modelo XGBoost para cuantiles
modelo = xgb.XGBRegressor(
    objective="reg:quantileerror",
    quantile_alpha=cuantiles,
    n_estimators=250,
    max_depth=5,
    enable_categorical=True,
    random_state=42
)

# 4. Entrenar el modelo con los datos hasta el 24 de enero
X_train = train[FEATURES]
y_train = train[TARGET]

modelo.fit(X_train, y_train)

print("✅ Modelo de cuantiles entrenado exitosamente.")

✅ Modelo de cuantiles entrenado exitosamente.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import numpy as np

# 1. Filtrar zonas con al menos 500 viajes (regla oficial)
zonas_seg = conteo_por_zona[conteo_por_zona >= 500].index.tolist()
trips_seg = trips[(trips['PULocationID'].isin(zonas_seg)) & (trips['trip_distance'] > 0) & (trips['fare_amount'] > 0)].copy()

# 2. Métricas de comportamiento por zona
df_feats = trips_seg.groupby('PULocationID').agg(
    viajes_dia=('tpep_pickup_datetime', lambda x: len(x) / 31),
    distancia_mediana=('trip_distance', 'median'),
    tarifa_mediana=('fare_amount', 'median'),
    propina_mediana=('tip_amount', 'median'),
    pct_fin_semana=('tpep_pickup_datetime', lambda x: (x.dt.dayofweek >= 5).mean()),
    pct_noche=('tpep_pickup_datetime', lambda x: (x.dt.hour.isin([22, 23, 0, 1, 2, 3, 4, 5])).mean())
).reset_index()

df_feats['log_viajes_dia'] = np.log1p(df_feats['viajes_dia'])

# 3. Escalado, K-Means y PCA
X_cols = ['log_viajes_dia', 'distancia_mediana', 'tarifa_mediana', 'propina_mediana', 'pct_fin_semana', 'pct_noche']
X_scaled = StandardScaler().fit_transform(df_feats[X_cols])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_scaled)
pca = PCA(n_components=2).fit(X_scaled)
coords = pca.transform(X_scaled)

df_feats['cluster'] = kmeans.labels_
df_feats['pca_1'] = coords[:, 0]
df_feats['pca_2'] = coords[:, 1]

# Nombrar grupos según su perfil
nombres = {0: "Barrios Periféricos", 1: "Vida Nocturna", 2: "Manhattan Central", 3: "Aeropuertos"}
df_feats['nombre_grupo'] = df_feats['cluster'].map(nombres)

# Unir con nombres reales de zonas
df_seg_final = df_feats.merge(zones, left_on='PULocationID', right_on='LocationID')
df_seg_final.to_csv("segmentacion_zonas.csv", index=False)

print("✅ Segmentación completada y guardada como segmentacion_zonas.csv")

✅ Segmentación completada y guardada como segmentacion_zonas.csv


In [ ]:
from google.colab import files

# 1. Guardar el modelo en formato liviano (.ubj)
modelo.save_model("modelo_pronostico.ubj")

# 2. Filtrar y guardar las filas de demanda necesarias para la API (25 enero a 1 febrero)
demanda_api = df_series[df_series['hora'] >= "2024-01-25 00:00:00"].copy()
demanda_api.to_parquet("demanda.parquet", index=False)

# 3. Descargar los tres archivos a tu computadora
print("⬇️ Descargando archivos...")
files.download("modelo_pronostico.ubj")
files.download("demanda.parquet")
files.download("segmentacion_zonas.csv")

⬇️ Descargando archivos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>